In [2]:
import pandas as pd
import numpy as np

# 1. Ingest Raw Files
fraud_df = pd.read_csv("../data/raw/Fraud_Data.csv")
ip_df = pd.read_csv("../data/raw/IpAddress_to_Country.csv")

# 2. Check Missing Values & Duplicates
print("Missing Values:\n", fraud_df.isnull().sum())
print("\nDuplicate Rows:", fraud_df.duplicated().sum())

# 3. Quantify Class Imbalance
counts = fraud_df['class'].value_counts()
pcts = fraud_df['class'].value_counts(normalize=True) * 100
print(f"\nClass Imbalance Profile:\nLegit (0): {counts[0]} ({pcts[0]:.2f}%)")
print(f"Fraud (1): {counts[1]} ({pcts[1]:.2f}%)")

# 4. Geolocation Range-Based Lookup
fraud_df['ip_address'] = fraud_df['ip_address'].astype(float)
ip_df['lower_bound_ip_address'] = ip_df['lower_bound_ip_address'].astype(float)
ip_df['upper_bound_ip_address'] = ip_df['upper_bound_ip_address'].astype(float)

# Sort both datasets to ensure binary boundary search works flawlessly
fraud_df = fraud_df.sort_values('ip_address')
ip_df = ip_df.sort_values('lower_bound_ip_address')

# Match the closest lower bound
merged_df = pd.merge_asof(
    fraud_df, ip_df,
    left_on='ip_address',
    right_on='lower_bound_ip_address',
    direction='backward'
)

# Enforce upper bound threshold rule
valid_ip_mask = (merged_df['ip_address'] >= merged_df['lower_bound_ip_address']) & \
                (merged_df['ip_address'] <= merged_df['upper_bound_ip_address'])

merged_df['country'] = np.where(valid_ip_mask, merged_df['country'], 'Unknown')
merged_df['country'] = merged_df['country'].fillna('Unknown')

# Drop range boundary helper columns
merged_df = merged_df.drop(columns=['lower_bound_ip_address', 'upper_bound_ip_address'])

# 5. Save Intermediate Geolocation Output to processed folder
merged_df.to_csv("../data/processed/fraud_with_country.csv", index=False)
print("\nTask 1 part 1 complete: Geolocation data integrated successfully!")


Missing Values:
 user_id           0
signup_time       0
purchase_time     0
purchase_value    0
device_id         0
source            0
browser           0
sex               0
age               0
ip_address        0
class             0
dtype: int64

Duplicate Rows: 0

Class Imbalance Profile:
Legit (0): 136961 (90.64%)
Fraud (1): 14151 (9.36%)

Task 1 part 1 complete: Geolocation data integrated successfully!
